<p align="center"><a href="https://www.plus2net.com/python/pandas-mysql-to_sql.php"><img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net Python Tutorials" width="300"></a></p>

# Pandas to_sql(): Export and Verify Database Records

**Pandas Tutorial Series — Input and Output**

[Full tutorial](https://www.plus2net.com/python/pandas-mysql-to_sql.php)

Run the SQLite examples immediately in Colab. The separate MySQL examples require a reachable MySQL server and credentials. SQLite practice illustrates the workflow; it does not verify MySQL behaviour.

## How to Use This Notebook

- Run cells from top to bottom and predict each output.
- Save a copy in Drive before changing examples.
- MySQL cells are disabled by default. Enable them only after setup.
- Use a dedicated practice database, never an existing production table.
- Synthetic data is embedded; no external download is required.
- Runtime files are temporary. Download the SQLite database through Colab's Files sidebar to keep it.

## What You Will Learn

Create and inspect tables, choose if_exists and index options, handle dates and missing values, import CSV/Excel, verify rows and values, and understand repeat-import risks.

## Start with a Runnable SQLite Example

<p>This in-memory database is recreated each time this cell runs. The later cells reuse its connection. SQLite needs no server, so it is convenient for learning in Colab.</p>

In [ ]:
import pandas as pd
import sqlite3
from io import StringIO

if "conn" in globals():
    conn.close()
conn = sqlite3.connect(":memory:")
df = pd.DataFrame({"student_id": [1, 2, 3], "name": ["Ravi", "Alex", "King"], "mark": [80, 70, 60]})
df.to_sql("student_demo", conn, if_exists="fail", index=False)
saved = pd.read_sql_query("SELECT * FROM student_demo ORDER BY student_id", conn)
pd.testing.assert_frame_equal(saved, df)
print(saved.to_string(index=False))
print("Rows and values verified.")

**Expected output**

```text
 student_id name  mark
          1 Ravi    80
          2 Alex    70
          3 King    60
Rows and values verified.
```

## Choose the to_sql Parameters

<div class="table-responsive"><table class="table table-bordered"><thead><tr><th scope="col">Parameter</th><th scope="col">Purpose</th></tr></thead><tbody>
<tr><td>name / con</td><td>Fixed table name and database engine or connection. Do not accept unchecked table names from user input.</td></tr>
<tr><td>if_exists</td><td>fail raises an error if a table exists; append inserts rows; replace drops and recreates the table. Newer Pandas also supports delete_rows, which removes records before inserting while retaining the existing table; check your installed version.</td></tr>
<tr><td>index / index_label</td><td>index=True writes the DataFrame index as a column; index_label names it. Usually choose index=False when a real ID column already exists.</td></tr>
<tr><td>dtype</td><td>SQLAlchemy types for SQLAlchemy connections; SQLite type strings for a sqlite3 connection.</td></tr>
<tr><td>chunksize / method</td><td>Batch size and insert strategy. method='multi' is driver-dependent; benchmark before claiming it is faster.</td></tr>
</tbody></table></div><p>Creating a SQL table from a DataFrame does not automatically establish primary keys or business constraints. Use an intentionally defined schema for durable applications.</p>

## Test fail, append and replace

<p>These operations affect only the disposable practice table. Append does not deduplicate or update existing records. Replace is not an UPDATE statement and can remove constraints and indexes.</p>

In [ ]:
try:
    df.to_sql("student_demo", conn, if_exists="fail", index=False)
except ValueError:
    print("fail: existing table was protected.")
new_student = pd.DataFrame({"student_id": [4], "name": ["Ron"], "mark": [65]})
new_student.to_sql("student_demo", conn, if_exists="append", index=False)
print("After append:", pd.read_sql_query("SELECT COUNT(*) AS n FROM student_demo", conn).iloc[0, 0])
df.to_sql("student_demo", conn, if_exists="replace", index=False)
print("After replacing disposable table:", pd.read_sql_query("SELECT COUNT(*) AS n FROM student_demo", conn).iloc[0, 0])

**Expected output**

```text
fail: existing table was protected.
After append: 4
After replacing disposable table: 3
```

## Use index_label When the Index Is Meaningful

<p>For an existing table, index=False omits the index column; the database then applies its defaults and constraints. It does not guarantee that NULL will be inserted.</p>

In [ ]:
indexed = df.set_index("student_id")
indexed.to_sql("indexed_demo", conn, if_exists="replace", index=True, index_label="student_id")
print(pd.read_sql_query("SELECT * FROM indexed_demo ORDER BY student_id", conn).to_string(index=False))

**Expected output**

```text
 student_id name  mark
          1 Ravi    80
          2 Alex    70
          3 King    60
```

## Convert Dates and Validate Before Writing

<p>Use an explicit input format. Coerce exposes invalid values as NaT; inspect them before export. Missing values normally become SQL NULL. Do not teach errors="ignore" as the current date-conversion approach.</p>

In [ ]:
dated = pd.DataFrame({"record_id": [1, 2], "c_date": ["01.10.2026", "invalid"]})
dated["c_date"] = pd.to_datetime(dated["c_date"], format="%d.%m.%Y", errors="coerce")
print("Invalid or missing dates:", int(dated["c_date"].isna().sum()))
valid_dates = dated.loc[dated["c_date"].notna()].copy()
valid_dates.to_sql("date_demo", conn, if_exists="replace", index=False)
print("Valid rows saved:", len(pd.read_sql_query("SELECT * FROM date_demo", conn)))

**Expected output**

```text
Invalid or missing dates: 1
Valid rows saved: 1
```

## Read CSV Data and Verify the Export

<p>The synthetic CSV is embedded so you can run this example without uploading a file. For your own CSV, upload it in Colab and use pd.read_csv("your_file.csv").</p>

In [ ]:
csv_text = "student_id,name,mark\n10,Ravi,80\n11,Alex,70\n"
csv_df = pd.read_csv(StringIO(csv_text))
assert csv_df["student_id"].is_unique
csv_df.to_sql("csv_demo", conn, if_exists="replace", index=False, chunksize=100)
csv_saved = pd.read_sql_query("SELECT * FROM csv_demo ORDER BY student_id", conn)
pd.testing.assert_frame_equal(csv_saved, csv_df)
print(csv_saved.to_string(index=False))

**Expected output**

```text
 student_id name  mark
         10 Ravi    80
         11 Alex    70
```

## Create a Downloadable SQLite Database

<p>This writes only to a dedicated demo file. Download it through the Colab Files sidebar; keeping a notebook does not keep its runtime files.</p>

In [ ]:
with sqlite3.connect("pandas_to_sql_practice.db") as file_conn:
    csv_df.to_sql("student_demo", file_conn, if_exists="replace", index=False)
    total = pd.read_sql_query("SELECT COUNT(*) AS n FROM student_demo", file_conn).iloc[0, 0]
print("Created pandas_to_sql_practice.db with", total, "rows.")

**Expected output**

```text
Created pandas_to_sql_practice.db with 2 rows.
```

## Connect to MySQL with SQLAlchemy

<p>Install <code>sqlalchemy</code> and <code>pymysql</code> in your environment, for example with <code>%pip install sqlalchemy pymysql</code> in Colab. The mysqlclient driver remains an alternative using mysql+mysqldb.</p><p>Create the database first. Set MYSQL_HOST, MYSQL_PORT, MYSQL_USER and MYSQL_DATABASE privately, or use Colab Secrets to obtain configuration. The password is prompted securely below. Never save credentials in notebook output.</p><p>Colab's localhost refers to its runtime, not your XAMPP computer. For local MySQL, run locally; for remote MySQL, use an authorised reachable server with the provider's required TLS configuration.</p>

In [ ]:
RUN_MYSQL = False

if RUN_MYSQL:
    import os
    from getpass import getpass
    from sqlalchemy import create_engine, URL, text
    
    url = URL.create(
        "mysql+pymysql",
        username=os.environ["MYSQL_USER"],
        password=getpass("MySQL password: "),
        host=os.environ["MYSQL_HOST"],
        port=int(os.environ.get("MYSQL_PORT", "3306")),
        database=os.environ["MYSQL_DATABASE"],
    )
    engine = create_engine(url, pool_pre_ping=True)
    with engine.connect() as connection:
        print("Connection test:", connection.execute(text("SELECT 1")).scalar_one())


This optional cell is disabled so **Run all** works without MySQL. Enable it only after completing setup. MySQL-specific examples were syntax checked, not executed against a MySQL server.

## Write and Verify a MySQL Practice Table

<p>Use a fresh demo table name. URL.create handles special characters in credentials without manual URL escaping. engine.begin commits on success and rolls back on exceptions where the database supports it; MySQL DDL can cause implicit commits, so do not promise rollback of table creation or replacement.</p>

In [ ]:
RUN_MYSQL = False

if RUN_MYSQL:
    from sqlalchemy import Integer, String
    
    mysql_df = pd.DataFrame({"student_id": [1, 2, 3], "name": ["Ravi", "Alex", "King"], "mark": [80, 70, 60]})
    with engine.begin() as connection:
        mysql_df.to_sql(
            "student_export_demo", connection, if_exists="fail", index=False,
            dtype={"student_id": Integer(), "name": String(80), "mark": Integer()},
            chunksize=100,
        )
    with engine.connect() as connection:
        stored = pd.read_sql_query(text("SELECT student_id, name, mark FROM student_export_demo ORDER BY student_id"), connection)
    assert stored["student_id"].tolist() == mysql_df["student_id"].tolist()
    assert stored["name"].tolist() == mysql_df["name"].tolist()
    assert stored["mark"].tolist() == mysql_df["mark"].tolist()
    print(stored.to_string(index=False))
    print("MySQL rows and values verified.")


This optional cell is disabled so **Run all** works without MySQL. Enable it only after completing setup. MySQL-specific examples were syntax checked, not executed against a MySQL server.

## Specify SQL Types for Measurements

<p>This retains the measurement-column example using a disposable table. Float is approximate; use an appropriate fixed-precision Numeric type for exact financial amounts and verify rounding requirements.</p>

In [ ]:
RUN_MYSQL = False

if RUN_MYSQL:
    from sqlalchemy import Float
    
    measurements = pd.DataFrame({"Sp_Mobile": [1.5, 2.5], "Sp_Desktop": [3.0, 4.0]})
    with engine.begin() as connection:
        measurements.to_sql("measurement_demo", connection, if_exists="fail", index=False,
                            dtype={"Sp_Mobile": Float(), "Sp_Desktop": Float()})


This optional cell is disabled so **Run all** works without MySQL. Enable it only after completing setup. MySQL-specific examples were syntax checked, not executed against a MySQL server.

## Excel to MySQL

<p>Upload employees.xlsx in Colab or place it beside your local script. Install openpyxl if needed. Inspect the sheet and validate required columns before writing; this example expects student_id, name and mark.</p>

In [ ]:
RUN_MYSQL = False

if RUN_MYSQL:
    excel_df = pd.read_excel("employees.xlsx", engine="openpyxl")
    required = ["student_id", "name", "mark"]
    if not set(required).issubset(excel_df.columns):
        raise ValueError("Excel file must include student_id, name and mark.")
    if excel_df["student_id"].isna().any() or not excel_df["student_id"].is_unique:
        raise ValueError("Student IDs must be present and unique in this file.")
    excel_df["mark"] = pd.to_numeric(excel_df["mark"], errors="raise")
    with engine.begin() as connection:
        excel_df[required].to_sql("excel_student_demo", connection, if_exists="fail", index=False)


This optional cell is disabled so **Run all** works without MySQL. Enable it only after completing setup. MySQL-specific examples were syntax checked, not executed against a MySQL server.

## CSV to MySQL

<p>Use the verified embedded CSV example above as the source. For production imports, compare IDs with the existing table and enforce unique constraints. Within-file uniqueness alone does not prevent a repeated import.</p>

In [ ]:
RUN_MYSQL = False

if RUN_MYSQL:
    with engine.begin() as connection:
        csv_df.to_sql("csv_student_demo", connection, if_exists="fail", index=False)
    with engine.connect() as connection:
        saved_csv = pd.read_sql_query(text("SELECT * FROM csv_student_demo ORDER BY student_id"), connection)
    print(saved_csv.to_string(index=False))


This optional cell is disabled so **Run all** works without MySQL. Enable it only after completing setup. MySQL-specific examples were syntax checked, not executed against a MySQL server.

## Update Existing Records without Replacing the Table

<p>Replace drops a table; it is not the right way to update marks in an established schema. Use parameterised SQL for record values. The following applies one update to the practice table; running it again adds another five marks.</p>

In [ ]:
RUN_MYSQL = False

if RUN_MYSQL:
    with engine.begin() as connection:
        connection.execute(text("UPDATE student_export_demo SET mark = mark + :bonus WHERE student_id = :student_id"),
                           {"bonus": 5, "student_id": 1})
    engine.dispose()


This optional cell is disabled so **Run all** works without MySQL. Enable it only after completing setup. MySQL-specific examples were syntax checked, not executed against a MySQL server.

## Common Problems and Checks

<ul><li>Access denied: check database credentials, database privileges and permitted host.</li><li>Connection refused or timeout: check server reachability; Colab cannot connect to your PC using localhost.</li><li>Unknown index column: use index=False or deliberately align the destination schema.</li><li>Duplicate key: check repeated imports and the destination's unique constraints; append is not an upsert.</li><li>Table already exists: fail intentionally protects it. Review before choosing append or a new table name.</li><li>Missing driver: install the driver matching the SQLAlchemy URL.</li><li>Unexpected dates or SQL types: validate conversions, define dtype and read values back.</li></ul><p>Do not treat the integer returned by to_sql as complete verification. Check destination rows, keys and values.</p>

## Practice Questions

<ol><li>Write a DataFrame without its index and read it back.</li><li>Append a new record, then explain what happens if you repeat the import.</li><li>Verify IDs and values, not just a printed success message.</li><li>Explain why replace can damage a production schema.</li><li>Create a file-based SQLite database and download it.</li><li>Identify which changes are needed to run the workflow against MySQL.</li></ol>

## Continue Learning

[Full tutorial](https://www.plus2net.com/python/pandas-mysql-to_sql.php)

[Input and Output pillar](https://www.plus2net.com/python/pandas-input-output.php)

[GitHub after upload](https://github.com/plus2net/pandas/blob/main/pandas_mysql_to_sql_practice.ipynb)

[Colab after GitHub upload](https://colab.research.google.com/github/plus2net/pandas/blob/main/pandas_mysql_to_sql_practice.ipynb)

**Validation:** SQLite worked examples were executed locally with Pandas 3.0.1. MySQL examples were syntax checked only; no MySQL server, browser or Colab execution was tested.

© 2026 plus2net.com